# Ecommerce Dataset Download

Downloads the Oct and Nov 2019 ecommerce behavior CSVs from Kaggle, unzips them, unions both months, and writes the result to a Delta table

In [0]:
%pip install kaggle -q
%restart_python

In [0]:
spark.table(f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}").count()

In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}").groupBy("event_time").count())

In [0]:
import os
import yaml
from kaggle.api.kaggle_api_extended import KaggleApi
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Load project config
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{config['volume']}"
DATASET = "mkechinov/ecommerce-behavior-data-from-multi-category-store"
TABLE = f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}"
PRODUCT_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_table']}"
MONTHS = ["2019-Oct.csv", "2019-Nov.csv"]

# Save your Kaggle credentials as Databricks Secrets
KAGGLE_SCOPE = config["kaggle_secret_scope"]
os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope=KAGGLE_SCOPE, key="kaggle_username")
os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope=KAGGLE_SCOPE, key="kaggle_api_key")


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{config['volume']}")
print(f"Schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {VOLUME}")

In [0]:
api = KaggleApi()
api.authenticate()

for month in MONTHS:
    api.dataset_download_file(DATASET, file_name=month, path=VOLUME)
    print(f"Downloaded {month}")

In [0]:
import zipfile
import os

for month in MONTHS:
    zip_path = f"{VOLUME}/{month}.zip"
    csv_path = f"{VOLUME}/{month}"
    if not os.path.exists(csv_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(VOLUME)
        print(f"Extracted {month}")
    else:
        print(f"{month} already extracted")

In [0]:
OCT, NOV = MONTHS

df_oct = spark.read.csv(f"{VOLUME}/{OCT}", header=True, inferSchema=True)
df_nov = spark.read.csv(f"{VOLUME}/{NOV}", header=True, inferSchema=True)

df = df_oct.union(df_nov)

final_df = (
    df
    .withColumn("_cat", F.coalesce(F.col("category_code"), F.lit("unknown")))
    .withColumn("_cat_parts", F.split("_cat", "\\."))
    .withColumn("category_l1", F.coalesce(F.expr("get(_cat_parts, 0)"), F.lit("unknown")))
    .withColumn("category_l2", F.coalesce(F.expr("get(_cat_parts, 1)"), F.lit("unknown")))
    .withColumn("category_l3", F.coalesce(F.expr("get(_cat_parts, 2)"), F.lit("unknown")))
    .withColumn("brand", F.coalesce(F.col("brand"), F.lit("unknown")))
    .withColumn("log_price", F.log1p(F.coalesce(F.col("price"), F.lit(0))))
    .withColumn("event_time_copy", F.col("event_time"))
    .drop("_cat", "_cat_parts")
)

final_df.write.mode("overwrite").saveAsTable(TABLE)
final_df = spark.table(TABLE)
display(final_df.limit(10))

Create a separate product dataset. This dataset contains high-level product features that don't require near-real-time transformations.

In [0]:
final_df = spark.table(TABLE)

In [0]:
spark.table(TABLE).count()

In [0]:
# ---------------------------------------------------------------------------
# Product catalog table — one row per product per day.
#
# The raw event table has ~110M rows, but most rows for the same product
# carry identical attributes. Deduplicating to the last observation per
# product per day:
#   - Handles attribute drift (price changes, recategorization, brand updates)
#   - Preserves event_time for point-in-time ColumnSelection joins
#   - Aligns with SlidingWindow(slide=1d) granularity
#   - Includes event_time_copy for item_recency (Max aggregation)
# ---------------------------------------------------------------------------
w_product_daily = Window.partitionBy(
    "product_id", F.to_date("event_time")
).orderBy(F.desc("event_time"))

product_df = (
    final_df
    .withColumn("_rn", F.row_number().over(w_product_daily))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        "product_id", "event_time",
        "price", "log_price", "brand",
        "category_l1", "category_l2", "category_l3",
        "category_id", "category_code",
    )
)

product_df.write.mode("overwrite").saveAsTable(PRODUCT_TABLE)
product_df = spark.table(PRODUCT_TABLE)

print(f"Full events:     {final_df.count():,}")
print(f"Product catalog: {product_df.count():,} (1 row per product per day)")
print(f"Distinct items:  {product_df.select('product_id').distinct().count():,}")
display(product_df.limit(10))